[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/02-calculus-and-optimization.ipynb)

# Module 2: Calculus and Optimization

Companion notebook to [Module 2](https://transformers-and-math.drmhse.com/modules/02-calculus-and-optimization) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

## Part 1 — A derivative is a prediction (section 2.1)
For $f(x) = x^2 e^x$, the derivative at 1 is $3e \approx 8.15$. Nudge $x$ and compare.

In [ ]:
f = lambda x: x ** 2 * math.exp(x)
for h in (1e-1, 1e-3, 1e-5):
    print(f'h={h:g}:  (f(1+h) - f(1)) / h = {(f(1 + h) - f(1)) / h:.5f}')
print('3e =', 3 * math.e)

## Part 2 — The chain rule, done by autograd (section 2.3)
PyTorch records each operation and applies the chain rule backwards.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
z = torch.tanh(3 * x)
z.backward()
print('autograd dz/dx =', x.grad.item(), '  by hand 3(1 - tanh²(3)) =', 3 * (1 - math.tanh(3) ** 2))

# Jacobian of y = xW is Wᵀ (rows of the Jacobian are outputs).
W = torch.randn(3, 2)
J = torch.autograd.functional.jacobian(lambda v: v @ W, torch.randn(3))
print('Jacobian equals Wᵀ:', torch.allclose(J, W.T))

## Part 3 — Gradient descent and the learning rate (section 2.4)
On the loss $\theta^2$ each step multiplies $\theta$ by $1 - 2\eta$.

In [ ]:
for eta in (0.1, 0.4, 0.9, 1.1):
    theta, path = 1.6, []
    for _ in range(15):
        path.append(theta)
        theta -= eta * 2 * theta
    plt.plot(path, label=f'η = {eta}')
plt.axhline(0, color='grey', lw=0.5); plt.ylim(-4, 4); plt.legend(); plt.title('θ over steps'); plt.show()

## Part 4 — Backpropagation by hand (section 2.5)
The worked example: $y = wx$, $a = \tanh(y)$, $L = (a - 1)^2$. Compute the backward pass manually, then let autograd check it.

In [ ]:
x, w = 2.0, 0.5
y = w * x; a = math.tanh(y); L = (a - 1) ** 2
dL_da = 2 * (a - 1)
dL_dy = dL_da * (1 - a ** 2)
dL_dw, dL_dx = dL_dy * x, dL_dy * w
print(f'by hand:  dL/dw = {dL_dw:.4f}   dL/dx = {dL_dx:.4f}')

xt = torch.tensor(x, requires_grad=True); wt = torch.tensor(w, requires_grad=True)
((torch.tanh(wt * xt) - 1) ** 2).backward()
print(f'autograd: dL/dw = {wt.grad.item():.4f}   dL/dx = {xt.grad.item():.4f}')

### Gradient check
The standard test of any hand-written backward pass: compare it with finite differences.

In [ ]:
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(2, 6), torch.nn.Tanh(), torch.nn.Linear(6, 2)).double()
inp = torch.randn(16, 2).double()
print('gradcheck passes:', torch.autograd.gradcheck(lambda i: net(i), (inp.requires_grad_(),)))

## Part 5 — Long chains vanish or explode (section 2.6)

In [ ]:
for factor in (0.8, 0.9, 1.0, 1.1):
    print(f'{factor}^50 = {factor ** 50:.3g}')

## Your turn
1. Use autograd to differentiate $p = e^a / (e^a + e^b)$ with respect to $a$ at $a = 1, b = 0$, and compare with $p(1 - p)$.
2. Find the largest $\eta$ for which gradient descent on $\theta^2$ still converges, by trying values.

<details><summary>Answers</summary>

1. Both give about 0.1966. 2. Anything below 1; at exactly 1, $\theta$ flips sign forever without shrinking.
</details>